Middleware

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
import os

# Initialize Groq LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"]
)

### Message-based summarization
agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [3]:
config = {
    "configurable": {
        "thread_id": "test-1"
    }
}

In [6]:
# Alternative test data
questions = [
    "What is 7 + 10 ?",
    "What is 11*4?",
    "What is 232/29?",
    "What is 24-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user\'s primary goal is to ask mathematical questions and receive answers.\n\n## SUMMARY\nThe user has asked a series of basic arithmetic questions, including addition, multiplication, division, and subtraction. The questions and answers are as follows: 2+2=4, 10*5=50, 100/4=25, 15-7=8, 3*3=9, 4*4=16, 7+10=17, 11*4=44, 232/29=8.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe user\'s next question is "What is 24-7?" which remains to be answered.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='636bee3b-703b-46ae-8d04-6fe879a7a895'), AIMessage(content='24 - 7 = 17.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 247, 'total_tokens': 256, 'completion_time': 0.00689226, 'completion_tokens_details': None, 'prompt_time': 0.013286924, 'prompt_tokens_details': None, 'queue_time': 0.161490485, 'total_

Token Size

In [6]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
import os

# Initialize Groq model
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"]
)

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 550),
            keep=("tokens", 200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [7]:
# Run test
cities = ["Delhi", "Bangalore", "Mumbai", "Lucknow", "Kolkata", "Chennai"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Delhi: ~258 tokens, 8 messages
[HumanMessage(content='Find hotels in Delhi', additional_kwargs={}, response_metadata={}, id='e93d3a2d-cb12-454e-80e6-cf76f42a5306'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8qj3re9vq', 'function': {'arguments': '{"city":"Delhi"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 226, 'total_tokens': 242, 'completion_time': 0.052266768, 'completion_tokens_details': None, 'prompt_time': 0.046080996, 'prompt_tokens_details': None, 'queue_time': 0.053438058, 'total_time': 0.098347764}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f17b4-4285-79d1-9cd7-bac507b80191-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Delhi'}, 'id': '8qj3re9vq', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

## Fraction


In [9]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
import os

# Initialize Groq model
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"]
)

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction", 0.005),  # 0.5%
            keep=("fraction", 0.002),     # 0.2%
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Delhi", "Bangalore", "Mumbai", "Lucknow", "Kolkata", "Chennai"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])

    # Groq model context window (approximate)
    fraction = tokens / 131072

    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response["messages"])

Delhi: ~59 tokens (0.0450%), 6 msgs
[HumanMessage(content='Hotels in Delhi', additional_kwargs={}, response_metadata={}, id='ea23c279-8051-425d-bcfb-fa023f8ab0b0'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'myyx5tecz', 'function': {'arguments': '{"city":"Delhi"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 210, 'total_tokens': 226, 'completion_time': 0.052466381, 'completion_tokens_details': None, 'prompt_time': 0.032266358, 'prompt_tokens_details': None, 'queue_time': 0.164753741, 'total_time': 0.084732739}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f17bb-0305-7c93-91a3-b1dfc731c517-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Delhi'}, 'id': 'myyx5tecz', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

## Human In the Loop MiddleWare

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [11]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
import os

# Initialize Groq model
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"]
)

agent = create_agent(
    model=llm,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ]
)

In [12]:

config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to adhi@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [13]:
result

{'messages': [HumanMessage(content="Send email to adhi@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='48c5f4e5-a48d-4c89-9ca5-6f4af474594f'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bjt4fpt8d', 'function': {'arguments': '{"body":"How are you?","recipient":"adhi@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 310, 'total_tokens': 343, 'completion_time': 0.085040351, 'completion_tokens_details': None, 'prompt_time': 0.104323024, 'prompt_tokens_details': None, 'queue_time': 0.161504539, 'total_time': 0.189363375}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f19e7-a24f-75a2-be5a-56bb4a87e17f-0', tool_calls=[{'name': 'send_email_tool', 'args': {'b

In [14]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been sent to adhi@test.com with the subject 'Hello' and body 'How are you?'.


In [15]:
result

{'messages': [HumanMessage(content="Send email to adhi@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='48c5f4e5-a48d-4c89-9ca5-6f4af474594f'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bjt4fpt8d', 'function': {'arguments': '{"body":"How are you?","recipient":"adhi@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 310, 'total_tokens': 343, 'completion_time': 0.085040351, 'completion_tokens_details': None, 'prompt_time': 0.104323024, 'prompt_tokens_details': None, 'queue_time': 0.161504539, 'total_time': 0.189363375}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f19e7-a24f-75a2-be5a-56bb4a87e17f-0', tool_calls=[{'name': 'send_email_tool', 'args': {'b